In [1]:
import sqlite3
import pandas as pd
import numpy as np
from scipy import stats

# Connect to database
connection = sqlite3.connect('cyber_security_logs.db')
print("Connected to cyber_security_logs.db")

Connected to cyber_security_logs.db


In [2]:
def query_db(sql):
    """Execute SQL query and return results as DataFrame"""
    try:
        return pd.read_sql_query(sql, connection)
    except Exception as e:
        print(f"SQL Error: {e}")
        return None

# SQL Query Notebook
Interactive SQL query environment for analyzing network traffic data from the cyber threat database.

In [3]:
query_db("""
SELECT "Source IP", COUNT(DISTINCT "Destination Port") as Unique_Ports_Targeted
FROM network_traffic
GROUP BY "Source IP"
HAVING Unique_Ports_Targeted > 5
ORDER BY Unique_Ports_Targeted DESC;
         """)

,"""Source IP""",Unique_Ports_Targeted
0,Source IP,24288


In [4]:
query_db("""
    SELECT "Source IP", COUNT(*) as Packet_Count
    FROM network_traffic
    GROUP BY "Source IP"
    ORDER BY Packet_Count DESC
    LIMIT 10;
""")

,"""Source IP""",Packet_Count
0,Source IP,283074


# Calculating Z-Scores for Network Features

In [7]:
# Pull the data into the notebook
# We use your query_db function to keep it clean
df_stats = query_db("SELECT * FROM network_traffic")

# Select numerical columns for analysis
# These are the 'signatures' of DDoS and Brute Force attacks
features = ['Flow Duration', 'Total Fwd Packets', 'Total Backward Packets']

print("Calculating Z-scores for network features...")

for col in features:
    z_col_name = f'{col}_zscore'
    # Calculate the absolute Z-score: |(x - mean) / std_dev|
    df_stats[z_col_name] = np.abs(stats.zscore(df_stats[col]))

# Identify the 3-sigma Outliers
# Logic: If ANY of our chosen features have a Z-score > 3, it's an anomaly
outliers = df_stats[
    (df_stats['Flow Duration_zscore'] > 3) | 
    (df_stats['Total Fwd Packets_zscore'] > 3)
]

print(f"Total Records Analyzed: {len(df_stats):,}")
print(f"Anomalous Records Identified (3-sigma): {len(outliers):,}")

# The 'Proof' - What did we actually catch?
print("\n--- Breakdown of Identified Outliers ---")
print(outliers['Label'].value_counts())

Calculating Z-scores for network features...
Total Records Analyzed: 283,074
Anomalous Records Identified (3-sigma): 9,721

--- Breakdown of Identified Outliers ---
Label
BENIGN              9371
DoS Hulk             317
DoS slowloris         22
DoS GoldenEye          6
DoS Slowhttptest       2
Heartbleed             1
PortScan               1
Infiltration           1
Name: count, dtype: int64
